# Parametric Populations

A single parametric `Stars` object can hold several stellar populations, e.g. a bulge and a disk, or the annuli of a radially resolved galaxy. Every population shares the same bin edges and only the masses differ, so they are stored along a leading population axis of the bin masses. This is the parametric analogue of the particles of a particle `Stars`: the integrated emission of every population is extracted in a single call, and the emission of each population can be extracted exactly as per particle emission is.

In this example we combine populations, give them their own parameters and morphologies, and get their emission and images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from unyt import Gyr, Msun, angstrom, kpc

from synthesizer.emission_models import PacmanEmission
from synthesizer.emission_models.attenuation import Calzetti2000
from synthesizer.grid import Grid
from synthesizer.instruments import FilterCollection, Instrument
from synthesizer.parametric import SFH, Annuli, Sersic2D, Stars, ZDist

grid = Grid("test_grid")

## Combining populations

Each population starts as an ordinary `Stars` object. Here an old, centrally concentrated bulge and a younger, extended and dustier disk.

In [ ]:
bulge = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.DelayedExponential(tau=0.5 * Gyr, max_age=8 * Gyr),
    metal_dist=ZDist.Normal(mean=0.02, sigma=0.005),
    initial_mass=5e10 * Msun,
    morphology=Sersic2D(r_eff=1 * kpc, sersic_index=4),
    tau_v=0.1,
)
disk = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=3 * Gyr),
    metal_dist=ZDist.Normal(mean=0.01, sigma=0.003),
    initial_mass=2e10 * Msun,
    morphology=Sersic2D(r_eff=4 * kpc, sersic_index=1, ellipticity=0.5),
    tau_v=0.6,
)

`Stars.from_populations` (or simply `bulge + disk`) combines them into one `Stars`. The populations are rebinned onto the union of their edges, so nothing is interpolated. Parameters that differ between the populations (here `tau_v`) become one value per population, and the morphologies become a `PerPopulation` morphology. Populations can be named and then looked up by name or index.

In [ ]:
stars = Stars.from_populations([bulge, disk], names=["bulge", "disk"])

print(f"Populations: {stars.npop} {stars.population_names}")
print(f"tau_v: {stars.tau_v}")
print(f"Morphology: {type(stars.morphology).__name__}")
print(f"Bulge mass: {stars['bulge'].initial_mass:.2e}")

## Emission

Populations with their own parameters (here `tau_v`) need a model with `per_particle=True`, exactly as particles with their own parameters do. It gives the emission of each population (one row per population, stored in `particle_spectra` like per particle emission) alongside the integrated emission, the sum over the populations, with each population attenuated by its own `tau_v`.

In [ ]:
model = PacmanEmission(grid, dust_curve=Calzetti2000())
per_pop_model = PacmanEmission(
    grid, dust_curve=Calzetti2000(), per_particle=True
)
stars.get_spectra(per_pop_model)
per_pop = stars.particle_spectra["emergent"]

separate = bulge.get_spectra(model).lnu + disk.get_spectra(model).lnu
print(
    "Matches the separate populations:",
    np.allclose(stars.spectra["emergent"].lnu, separate),
)

In [ ]:
fig, ax = plt.subplots()
for name, lnu in zip(stars.population_names, per_pop.lnu):
    ax.loglog(per_pop.lam, lnu, label=name)
ax.loglog(per_pop.lam, stars.spectra["emergent"].lnu, "k--", label="Total")
ax.set_xlim(1e3, 1e5)
ax.set_ylim(1e26, None)
ax.set_xlabel(r"$\lambda / \AA$")
ax.set_ylabel(r"$L_\nu / \mathrm{erg\,s^{-1}\,Hz^{-1}}$")
ax.legend()
plt.show()

## Images

Since each population has its own morphology, an image is the sum of each population's image, which needs the photometry of each population (`get_particle_photo_lnu`).

In [ ]:
filters = FilterCollection(
    tophat_dict={
        "UV": {"lam_eff": 1500 * angstrom, "lam_fwhm": 300 * angstrom},
        "R": {"lam_eff": 6500 * angstrom, "lam_fwhm": 1000 * angstrom},
    },
    new_lam=grid.lam,
)
imager = Instrument("imager", filters=filters, resolution=0.2 * kpc)

stars.get_particle_photo_lnu(filters)
images = stars.get_images_luminosity(
    "emergent", fov=30 * kpc, instrument=imager
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, band in zip(axes, ("UV", "R")):
    ax.imshow(np.log10(images[band].arr), origin="lower", cmap="magma")
    ax.text(0.05, 0.92, band, color="w", transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

The disk's young stars dominate the UV image while the bulge stands out in the red one.

## Annuli

A radially resolved galaxy can instead share a single profile split into annuli, one population per annulus, with the `Annuli` morphology. Each annulus holds only the light of its population within that annulus. Here the stellar populations get older towards the centre.

In [ ]:
radii = np.array([0, 1, 2, 4, 8, np.inf]) * kpc
annuli_pops = [
    Stars(
        grid.log10ages,
        grid.metallicities,
        sf_hist=SFH.Constant(max_age=max_age),
        metal_dist=0.01,
        initial_mass=1e10 * Msun,
        tau_v=0.3,
    )
    for max_age in (10, 5, 2, 0.5, 0.1) * Gyr
]
annuli = Stars.from_populations(annuli_pops)
annuli.morphology = Annuli(Sersic2D(r_eff=3 * kpc, sersic_index=1), radii)

annuli.get_spectra(per_pop_model)
annuli.get_particle_photo_lnu(filters)
images = annuli.get_images_luminosity(
    "emergent", fov=30 * kpc, instrument=imager
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, band in zip(axes, ("UV", "R")):
    ax.imshow(np.log10(images[band].arr), origin="lower", cmap="magma")
    ax.text(0.05, 0.92, band, color="w", transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

Morphologies are integrated over each pixel and normalised by the profile's total light, so light falling outside the field of view is lost rather than added back into the image. The outer annulus here extends beyond the image, so the image holds slightly less light than the populations emit.

In [ ]:
total = annuli.particle_photo_lnu["emergent"]["R"].value.sum()
print(
    "Fraction of the R band light in the image:",
    f"{np.asarray(images['R'].arr).sum() / total:.4f}",
)